# SMAP Missing-Data Ablation for the 2026 ECE Deployment

This experiment measures how the confirmed SMAP zero-fill defect affected ECE predictions and compares four operational corrections against the original zero-filled policy. All imputations are derived strictly from the `derived_8.4` training pool, and the ECE target is used only for evaluation.


## Load the versioned experiment implementation

The notebook imports the tracked runner rather than duplicating model logic. Paths resolve from the experiment directory so the same code can be executed from a clean checkout after Git LFS data are hydrated.

In [1]:
from pathlib import Path
import importlib.util
import pandas as pd

EXP_DIR = Path.cwd()
RUNNER_PATH = EXP_DIR / "run_ablation.py"
spec = importlib.util.spec_from_file_location("smap_ablation", RUNNER_PATH)
ablation = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ablation)
config, parent_config = ablation.load_configuration()
print(f"Experiment: {config['experiment']['name']}")
print(f"Strategies: {', '.join(config['strategies'])}")


Experiment: derived_8.4-ece-smap-ablation-1.0
Strategies: zero_filled_existing_policy, native_missing_existing_training, training_month_climatology, no_smap_retrained, block_masked_retrained


## Audit inputs before fitting models

This check verifies row alignment, counts the selected SMAP features, and contrasts the parent zero-filled ECE split with the corrected native-missing split before any model sees the data.

In [2]:
data = ablation.load_data(config, parent_config)
smap_features = data["smap_features"]
print(f"Training rows: {len(data['train']):,}")
print(f"ECE rows: {len(data['native']):,}")
print(f"Selected features: {len(data['features'])}")
print(f"Selected SMAP features: {len(smap_features)}")
print(f"Zero-policy finite SMAP cells: {int(data['zero'][smap_features].notna().sum().sum())}")
print(f"Native-missing finite SMAP cells: {int(data['native'][smap_features].notna().sum().sum())}")

Training rows: 14,608
ECE rows: 150
Selected features: 38
Selected SMAP features: 6
Zero-policy finite SMAP cells: 650
Native-missing finite SMAP cells: 0


## Fit five-seed policy ablations without ECE-target leakage

The standard weighted model is trained once per seed and evaluated against zero-filled, native-missing, and training-month climatology inputs. Separate models are retrained without SMAP and with deterministic 30-day station blocks masked in 20% of training blocks; the 20% rate is an explicit simulation parameter recorded in `config.yaml`, not a tuned result.

In [3]:
from pathlib import Path
import subprocess
import sys

EXP_DIR = Path.cwd()
RUNNER_PATH = EXP_DIR / "run_ablation.py"
completed = subprocess.run(
    [sys.executable, str(RUNNER_PATH)],
    cwd=EXP_DIR,
    check=True,
    capture_output=True,
    text=True,
)
print(completed.stdout)


SMAP ABLATION INPUT AUDIT
train_rows=14608 ece_rows=150 features=38 smap_features=6
zero_policy_finite_smap=650
native_missing_finite_smap=0
climatology_remaining_missing=0
seed=42 complete
seed=7 complete
seed=13 complete
seed=101 complete
seed=123 complete

SMAP ABLATION SUMMARY
                        strategy  rmse_mean  rmse_std  mae_mean  bias_mean  ubrmse_mean   r2_mean  pearson_r_mean  rmse_change_vs_zero
     zero_filled_existing_policy   0.074504  0.007252  0.067093  -0.055493     0.049306 -1.526627        0.089987             0.000000
native_missing_existing_training   0.072539  0.003958  0.065026  -0.054195     0.048075 -1.382752        0.117342            -0.001965
      training_month_climatology   0.069335  0.007528  0.062513  -0.048486     0.048974 -1.192240        0.087492            -0.005169
               no_smap_retrained   0.070625  0.004369  0.063337  -0.050787     0.048976 -1.260177        0.104094            -0.003879
          block_masked_retrained   0.070121

## Compare physical errors across missing-data policies

The primary comparison uses pooled RMSE, MAE, bias, and unbiased RMSE across five fixed seeds. R-squared and Pearson correlation remain secondary because the ECE targets have unusually low within-station variance.


In [4]:
from pathlib import Path
import pandas as pd

EXP_DIR = Path.cwd()
summary = pd.read_csv(EXP_DIR / "summary.csv")
seed_metrics = pd.read_csv(EXP_DIR / "seed_metrics.csv")
station_metrics = pd.read_csv(EXP_DIR / "station_metrics.csv")
print(summary.to_string(index=False, float_format=lambda value: f"{value:.6f}"))


                        strategy  rmse_mean  rmse_std  mae_mean  bias_mean  ubrmse_mean   r2_mean  pearson_r_mean  rmse_change_vs_zero
     zero_filled_existing_policy   0.074504  0.007252  0.067093  -0.055493     0.049306 -1.526627        0.089987             0.000000
native_missing_existing_training   0.072539  0.003958  0.065026  -0.054195     0.048075 -1.382752        0.117342            -0.001965
      training_month_climatology   0.069335  0.007528  0.062513  -0.048486     0.048974 -1.192240        0.087492            -0.005169
               no_smap_retrained   0.070625  0.004369  0.063337  -0.050787     0.048976 -1.260177        0.104094            -0.003879
          block_masked_retrained   0.070121  0.011260  0.062790  -0.048459     0.049594 -1.267034        0.079917            -0.004384


## Check whether improvements are consistent across seeds and stations

Mean improvements can hide seed instability or a single dominant station. The next table reports paired seed-level RMSE deltas against zero-fill and station-level mean RMSE for each strategy, without treating five seeds as a substitute for independent field deployments.

In [5]:
from pathlib import Path
import pandas as pd

EXP_DIR = Path.cwd()
seed_metrics = pd.read_csv(EXP_DIR / "seed_metrics.csv")
station_metrics = pd.read_csv(EXP_DIR / "station_metrics.csv")
zero = seed_metrics.loc[
    seed_metrics["strategy"].eq("zero_filled_existing_policy"),
    ["seed", "rmse"],
].rename(columns={"rmse": "zero_rmse"})
paired = seed_metrics.merge(zero, on="seed")
paired["rmse_delta_vs_zero"] = paired["rmse"] - paired["zero_rmse"]
paired_summary = paired.groupby("strategy", sort=False).agg(
    mean_delta=("rmse_delta_vs_zero", "mean"),
    improved_seeds=("rmse_delta_vs_zero", lambda values: int((values < 0).sum())),
    total_seeds=("seed", "nunique"),
).reset_index()
print("PAIRED SEED RMSE DELTAS")
print(paired_summary.to_string(index=False, float_format=lambda value: f"{value:.6f}"))

station_summary = station_metrics.groupby(
    ["station_id", "strategy"], sort=False
)["rmse"].mean().unstack("strategy")
print("\nSTATION-LEVEL MEAN RMSE")
print(station_summary.to_string(float_format=lambda value: f"{value:.6f}"))

PAIRED SEED RMSE DELTAS
                        strategy  mean_delta  improved_seeds  total_seeds
     zero_filled_existing_policy    0.000000               0            5
native_missing_existing_training   -0.001965               3            5
      training_month_climatology   -0.005169               5            5
               no_smap_retrained   -0.003879               3            5
          block_masked_retrained   -0.004384               4            5

STATION-LEVEL MEAN RMSE
strategy                 zero_filled_existing_policy  native_missing_existing_training  training_month_climatology  no_smap_retrained  block_masked_retrained
station_id                                                                                                                                                   
ECE_BBG_Lost_Meadow                         0.072933                          0.067974                    0.069033           0.073044                0.071875
ECE_BBG_Main_St                  

## Interpretation and operational recommendation

The final synthesis separates what this ablation establishes from what remains uncertain. The recommendation prioritizes the best mean physical error while retaining a fallback that does not depend on synthetic SMAP values.

In [6]:
summary = pd.read_csv(Path.cwd() / "summary.csv")
summary = summary.sort_values("rmse_mean")
best = summary.iloc[0]
zero = summary.loc[summary["strategy"].eq("zero_filled_existing_policy")].iloc[0]
relative_gain = 100 * (zero["rmse_mean"] - best["rmse_mean"]) / zero["rmse_mean"]
print("FINAL INTERPRETATION")
print(f"Best mean-RMSE policy: {best['strategy']}")
print(f"RMSE improvement vs zero-fill: {zero['rmse_mean'] - best['rmse_mean']:.6f} m3/m3 ({relative_gain:.2f}%)")
print("Native NaN alone improves the original policy, confirming that zero-fill was harmful.")
print("The remaining negative bias is much larger than the SMAP-policy gain, so sensor/domain shift remains the primary unresolved error source.")
print("Recommendation: use training-only monthly climatology with missingness provenance for this model family, and retain the no-SMAP model as the operational fallback for persistently invalid urban footprints.")

FINAL INTERPRETATION
Best mean-RMSE policy: training_month_climatology
RMSE improvement vs zero-fill: 0.005169 m3/m3 (6.94%)
Native NaN alone improves the original policy, confirming that zero-fill was harmful.
The remaining negative bias is much larger than the SMAP-policy gain, so sensor/domain shift remains the primary unresolved error source.
Recommendation: use training-only monthly climatology with missingness provenance for this model family, and retain the no-SMAP model as the operational fallback for persistently invalid urban footprints.
